# Generation: dcGAN
- Used the '30kds_real' dataset. This dataset has 2 variations. One with central crop and other with a more robust crop method explained on the 'create_30k_real_yolo_crop.py' file
- Images are all with size 128x128px
- Runs on gpu/cpu

In [1]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.utils import save_image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import torchvision.utils as vutils
import csv

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Parameters
batch_size = 64
latent_dim = 100
num_epochs = 250
learning_rate = 0.0002
beta1 = 0.5
num_train_images = 0  # Number of training images to use (0 to use all)
dataset_path = "30kds_real_face_crop_dlib"
save_interval = 10

# Ensure directories exist for saving outputs
base_dir = os.getcwd()  # Gets the current working directory in Jupyter Notebook
model_dir = gen_images_dir = os.path.join(base_dir, "generators/dcGAN")
os.makedirs(gen_images_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# Prepare CSV file to log losses
loss_log_path = os.path.join(model_dir, "training_losses.csv")
with open(loss_log_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['epoch', 'discriminator_Loss', 'generator_Loss'])

Using device: cuda


In [2]:
# Data Loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),  # match Tanh range (-1 to 1)
])

# Load the dataset
full_dataset = ImageFolder(root=dataset_path, transform=transform)

# Randomly select a subset of images
total_images = len(full_dataset)
if num_train_images == 0:
    dataset = full_dataset
    print(f"Full dataset loaded.")
elif total_images > num_train_images:
    indices = torch.randperm(total_images)[:num_train_images].tolist()
    dataset = Subset(full_dataset, indices)
    print(f"Loaded {num_train_images} images.")
else:
    dataset = full_dataset
    print(f"Warning: Requested {num_train_images} images but only {total_images} are available.")
    print(f"Full dataset loaded.")

# Create data loader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

Full dataset loaded.


In [3]:
# Generator Model (for 128x128px images)
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 1024, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(True),
            nn.ConvTranspose2d(1024, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

# Discriminator Model
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        
        self.main = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)
    
generator = Generator().to(device)
discriminator = Discriminator().to(device)

# Weight initialization
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

generator.apply(weights_init)
discriminator.apply(weights_init)

Discriminator(
  (main): Sequential(
    (0): Conv2d(3, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (12): BatchNorm2d(512, eps=1e-05, mom

In [4]:
# Loss and optimizers
criterion = nn.BCELoss()
g_optimizer = optim.Adam(generator.parameters(), lr=learning_rate, betas=(beta1, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(beta1, 0.999))

# Fixed noise for visualization
fixed_noise = torch.randn(16, latent_dim, 1, 1, device=device)

In [5]:
def train_gan(generator, discriminator, dataloader, criterion, g_optimizer, d_optimizer, 
              num_epochs, device, latent_dim, save_interval, gen_images_dir, loss_log_path):
    # Fixed noise for visualization
    fixed_noise = torch.randn(16, latent_dim, 1, 1, device=device)
    
    print("Starting training...")
    
    for epoch in range(num_epochs):
        # Initialize epoch stats
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0
        num_batches = 0
        last_batch_d_loss = 0.0
        last_batch_g_loss = 0.0
        
        # Create progress bar for better visibility
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        # Batch training loop
        for i, (images, _) in enumerate(pbar):
            batch_size = images.size(0)
            real_images = images.to(device)
            
            # Labels
            real_labels = torch.ones(batch_size, device=device)
            fake_labels = torch.zeros(batch_size, device=device)
            
            # Train Discriminator
            d_optimizer.zero_grad()
            real_outputs = discriminator(real_images)
            d_real_loss = criterion(real_outputs, real_labels)
            noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
            with torch.no_grad():
                fake_images = generator(noise)
            fake_outputs = discriminator(fake_images)
            d_fake_loss = criterion(fake_outputs, fake_labels)
            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            d_optimizer.step()
            
            # Train Generator
            g_optimizer.zero_grad()
            noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
            fake_images = generator(noise)
            fake_outputs = discriminator(fake_images)
            g_loss = criterion(fake_outputs, real_labels)
            g_loss.backward()
            g_optimizer.step()

            last_batch_d_loss = d_loss.item()
            last_batch_g_loss = g_loss.item()
            epoch_d_loss += last_batch_d_loss
            epoch_g_loss += last_batch_g_loss
            num_batches += 1
            
            # Update progress bar with current batch loss
            pbar.set_postfix({
                'D Loss': f"{last_batch_d_loss:.4f}",
                'G Loss': f"{last_batch_g_loss:.4f}"
            })
        
        # Log losses to CSV
        with open(loss_log_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch + 1, epoch_d_loss, epoch_g_loss])
        
        # Print current epoch loss
        print(f"Epoch [{epoch+1}/{num_epochs}] - D Loss: {last_batch_d_loss:.4f}, G Loss: {last_batch_g_loss:.4f}")
        
        # Generate and save images at checkpoints
        if (epoch + 1) % save_interval == 0 or epoch == 0:
            try:
                with torch.no_grad():
                    fake_samples = generator(fixed_noise).detach().cpu()
                    grid = vutils.make_grid(fake_samples, nrow=4, padding=2, normalize=True)
                    save_path = os.path.join(gen_images_dir, f"epoch_{epoch+1}.png")
                    save_image(grid, save_path)
                    print(f"Generated samples saved to {save_path}")
            except Exception as e:
                print(f"Error during saving: {str(e)}")
    
    # Save final models
    try:
        torch.save(generator.state_dict(), os.path.join(os.path.dirname(gen_images_dir), "generator_final.pth"))
        torch.save(discriminator.state_dict(), os.path.join(os.path.dirname(gen_images_dir), "discriminator_final.pth"))
        print("Final models saved successfully")
    except Exception as e:
        print(f"Error saving final models: {str(e)}")

train_gan(
    generator=generator,
    discriminator=discriminator,
    dataloader=dataloader,
    criterion=criterion,
    g_optimizer=g_optimizer,
    d_optimizer=d_optimizer,
    num_epochs=num_epochs,
    device=device,
    latent_dim=latent_dim,
    save_interval=save_interval,
    gen_images_dir=gen_images_dir,
    loss_log_path=loss_log_path
)

Starting training...


Epoch 1/250: 100%|██████████| 405/405 [00:28<00:00, 14.25it/s, D Loss=1.0452, G Loss=2.0477] 


Epoch [1/250] - D Loss: 1.0452, G Loss: 2.0477
Generated samples saved to c:\Users\danie\Desktop\UNI-4ano\DL\generators/dcGAN\epoch_1.png


Epoch 2/250: 100%|██████████| 405/405 [00:28<00:00, 14.41it/s, D Loss=0.9232, G Loss=2.6087]


Epoch [2/250] - D Loss: 0.9232, G Loss: 2.6087


...

Epoch [248/250] - D Loss: 0.0021, G Loss: 8.0907


Epoch 249/250: 100%|██████████| 405/405 [00:27<00:00, 14.63it/s, D Loss=0.6321, G Loss=2.4251] 


Epoch [249/250] - D Loss: 0.6321, G Loss: 2.4251


Epoch 250/250: 100%|██████████| 405/405 [00:27<00:00, 14.74it/s, D Loss=0.5302, G Loss=0.3299] 

Epoch [250/250] - D Loss: 0.5302, G Loss: 0.3299
Generated samples saved to c:\Users\danie\Desktop\UNI-4ano\DL\generators/dcGAN\epoch_250.png
Final models saved successfully


In [6]:
# Generate final images
with torch.no_grad():
    sample_noise = torch.randn(64, latent_dim, 1, 1, device=device)
    sample_images = generator(sample_noise).detach().cpu()

    grid = vutils.make_grid(sample_images, nrow=8, padding=2, normalize=True)
    final_sample_path = os.path.join(gen_images_dir, "final_samples.png")
    save_image(grid, final_sample_path)

print(f"Final samples saved to {final_sample_path}")

Final samples saved to c:\Users\danie\Desktop\UNI-4ano\DL\generators/dcGAN\final_samples.png
